# Анализ вечной мерзлоты и прогнозирование чрезвычайных ситуаций в Якутии

**Назначение ноутбука:** показать экзаменационной комиссии, зачем выполнено исследование, как была построена модель и какой практически применимый результат получен для планирования реакции органов власти и спасательных служб.

Объект исследования: населенные пункты криолитозоны Якутии и гидрологическая система реки Лены.  
Фокус: деградация многолетнемерзлых грунтов, весенние половодья/заторы, риск повреждения инфраструктуры и логика реагирования МЧС.


## 1. Короткий ответ комиссии: что сделано

В работе собран расчетный комплекс, который переводит разнородные наблюдения в управленческий сигнал риска:

1. Метеорологические и геокриологические данные преобразованы в физические индексы: `DDT`, `DDF`, высота снега, оттепели, ROS-события и глубина сезонного протаивания `ALT`.
2. Построены модели деградации мерзлоты: классическая модель Стефана, гибридная модель со снежной теплоизоляцией и многофакторная регрессия.
3. Для реки Лены рассчитаны лаги прохождения весеннего пика, чтобы получить окно упреждения для Якутска.
4. Разработана СППР: интегральный балл `PSRS` переводит физические параметры в уровни реагирования: зеленый, желтый, красный.

**Итоговый результат:** не просто графики, а воспроизводимая схема принятия решения: от данных мониторинга до рекомендаций для спасательных служб и муниципального управления.


## 2. План реализации исследования

| Этап | Что делаем | Что получает комиссия |
|---|---|---|
| 1 | Загружаем подготовленные таблицы и проверяем состав данных | Понятный корпус исходных признаков |
| 2 | Описываем физические индексы риска | Связь данных с процессами мерзлоты и паводка |
| 3 | Демонстрируем модели `ALT`, глубинного потепления и гидрологических лагов | Математическую основу прогноза |
| 4 | Собираем интегральный балл `PSRS` | Правило перехода от чисел к уровню риска |
| 5 | Показываем сценарии и рекомендации | Практический результат для МЧС/органов власти |


## 3. Подготовка окружения

Ноутбук сделан устойчивым для защиты: основные демонстрационные расчеты используют только стандартную библиотеку Python. Это снижает риск, что на чужом компьютере защита остановится из-за отсутствия `pandas`, `numpy`, `sklearn` или `matplotlib`.


In [ ]:
from pathlib import Path
import csv
import math
import statistics
from collections import Counter

PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / "data_tables").exists() and (PROJECT_DIR / "plots").exists():
    RESULTS_DIR = PROJECT_DIR
elif (PROJECT_DIR / "RESULTS (Read me, Dara)").exists():
    RESULTS_DIR = PROJECT_DIR / "RESULTS (Read me, Dara)"
elif (PROJECT_DIR.parent / "RESULTS (Read me, Dara)").exists():
    RESULTS_DIR = PROJECT_DIR.parent / "RESULTS (Read me, Dara)"
else:
    raise FileNotFoundError("Не найден каталог 'RESULTS (Read me, Dara)' с таблицами и графиками")

DATA_DIR = RESULTS_DIR / "data_tables"
PLOTS_DIR = RESULTS_DIR / "plots"

def load_csv(filename):
    with open(DATA_DIR / filename, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def to_float(value, default=None):
    try:
        if value in (None, ""):
            return default
        return float(str(value).replace(",", "."))
    except ValueError:
        return default

print("Каталог результатов:", RESULTS_DIR)
print("Каталог таблиц:", DATA_DIR)
print("Каталог графиков:", PLOTS_DIR)


Каталог результатов: /workspaces/permafrost_analysis_dara/RESULTS (Read me, Dara)
Каталог таблиц: /workspaces/permafrost_analysis_dara/RESULTS (Read me, Dara)/data_tables
Каталог графиков: /workspaces/permafrost_analysis_dara/RESULTS (Read me, Dara)/plots


## 4. Состав расчетных данных

В каталог результатов вынесены только подготовленные таблицы, которые нужны для воспроизведения выводов. Ниже проверяется, что таблицы доступны и имеют ненулевой объем.


In [ ]:
tables = {
    "mchs_events.csv": load_csv("mchs_events.csv"),
    "mchs_events_lena_bank.csv": load_csv("mchs_events_lena_bank.csv"),
    "daily_anomalies_operational.csv": load_csv("daily_anomalies_operational.csv"),
    "monthly_anomalies_operational.csv": load_csv("monthly_anomalies_operational.csv"),
    "scenario_annual.csv": load_csv("scenario_annual.csv"),
    "anomaly_chs_linked.csv": load_csv("anomaly_chs_linked.csv"),
    "water_level_annual_features.csv": load_csv("water_level_annual_features.csv"),
    "water_level_monthly_corr.csv": load_csv("water_level_monthly_corr.csv"),
    "water_level_peak_lags_yakutsk.csv": load_csv("water_level_peak_lags_yakutsk.csv"),
}

for name, rows in tables.items():
    print(f"{name:36s} | строк: {len(rows):4d} | столбцов: {len(rows[0]) if rows else 0:2d}")


mchs_events.csv                      | строк:   79 | столбцов:  7
mchs_events_lena_bank.csv            | строк:   17 | столбцов:  9
daily_anomalies_operational.csv      | строк: 3095 | столбцов: 15
monthly_anomalies_operational.csv    | строк:  163 | столбцов: 16
scenario_annual.csv                  | строк:   16 | столбцов: 10
anomaly_chs_linked.csv               | строк:    5 | столбцов:  9
water_level_annual_features.csv      | строк:  144 | столбцов: 22
water_level_monthly_corr.csv         | строк:    9 | столбцов: 10
water_level_peak_lags_yakutsk.csv    | строк:   16 | столбцов:  9


## 5. Исходная логика признаков

Модель не пытается угадывать ЧС как черный ящик. Каждый показатель имеет физическую интерпретацию:

| Показатель | Смысл |
|---|---|
| `DDT` | накопленное летнее тепло, определяет потенциал протаивания |
| `DDF` | накопленный зимний холод, отражает потенциал промерзания |
| `H_snow` | высота снега, теплоизолирует грунт зимой |
| `ROS` | дождь по снегу: оттепель + осадки + снежный покров |
| `ALT` | глубина сезонного протаивания, ключевой геотехнический показатель |
| уровни воды | гидрологическая нагрузка на населенные пункты Лены |

Главная идея: совместить медленный стратегический риск мерзлоты и быстрый оперативный риск половодья.


In [ ]:
lena = tables["mchs_events_lena_bank.csv"]
daily = tables["daily_anomalies_operational.csv"]
monthly = tables["monthly_anomalies_operational.csv"]
scenario = tables["scenario_annual.csv"]
linked = tables["anomaly_chs_linked.csv"]
water = tables["water_level_annual_features.csv"]
yakutsk_water = [r for r in water if r.get("post") == "Якутск"]

print(f"Архив ЧС по всей базе: {len(tables['mchs_events.csv'])} записей.")
print(f"ЧС на Лене/береговой зоне: {len(lena)} записей; годы: {min(int(r['year']) for r in lena)}-{max(int(r['year']) for r in lena)}.")
print("Типы ЧС на Лене:", dict(Counter(r["type_chs"] for r in lena)))
print("Суточные аномалии:", dict(Counter(r["severity"] for r in daily)))
print("Месячные аномалии:", dict(Counter(r["severity"] for r in monthly)))
print("Сценарные годы:", dict(Counter(r["сценарий"] for r in scenario)))
print(f"Связанные комплексные аномалии и ЧС: {sum(1 for r in linked if r['была_ли_ЧС'] == 'True')} из {len(linked)} случаев.")
print(f"Якутск, весенние пики воды: максимум {max(to_float(r['spring_peak_cm']) for r in yakutsk_water):.0f} см; среднее {statistics.mean(to_float(r['spring_peak_cm']) for r in yakutsk_water):.1f} см.")


Архив ЧС по всей базе: 79 записей.
ЧС на Лене/береговой зоне: 17 записей; годы: 2008-2023.
Типы ЧС на Лене: {'Половодье': 7, 'Затор': 9, 'Паводок': 1}
Суточные аномалии: {'умеренная': 2600, 'экстремальная': 495}
Месячные аномалии: {'экстремальная': 42, 'умеренная': 121}
Сценарные годы: {'Базовый': 6, 'Умеренный': 5, 'Высокий': 5}
Связанные комплексные аномалии и ЧС: 3 из 5 случаев.
Якутск, весенние пики воды: максимум 854 см; среднее 648.6 см.


## 6. Модель глубины сезонного протаивания `ALT`

Используются три уровня сложности, чтобы комиссия увидела и физический смысл, и прирост качества:

**Классическая модель Стефана**

$$ALT = E \cdot \sqrt{DDT}, \quad E = 4.3448$$

**Гибридная модель Стефана со снежной теплоизоляцией**

$$ALT = E_{base} \cdot \sqrt{DDT} \cdot (1 + eta H_{snow}), \quad E_{base}=4.1866, \; eta=0.00155$$

**Многофакторная регрессия**

$$ALT = 197.99 + 0.0096DDT - 0.0051DDF + 0.370H_{snow} - 0.0045P_{summer}$$

Интерпретация: теплое лето увеличивает протаивание, высокий снег ослабляет зимнее выхолаживание, а летние осадки могут снижать прогрев через испарительное охлаждение и изменение теплового режима поверхности.


![Сравнение моделей ALT](plots/ALT_Models_Comparison.png)


## 7. Тренд глубинного потепления мерзлоты

Для стратегического риска важен не только сезонный слой, но и температура на глубине около 3.2 м, где сглажены сезонные колебания. В презентационной форме тренд можно записать относительно 2024 года:

$$T_{320}(y) = -0.126 + 0.0215 \cdot (y - 2024)$$

Это соответствует скорости примерно **+0.215 °C за десятилетие**. При линейной экстраполяции пересечение с нулевой отметкой приходится примерно на **2030 год**, то есть на горизонт, важный для планирования обследований, ремонта и термостабилизации оснований.


![Тренд потепления мерзлоты](plots/Deep_Permafrost_Warming_Trend.png)


## 8. Гидрологический контур: Лена и окно упреждения

Для МЧС особенно важен не сам факт высокого уровня воды, а возможность подготовиться заранее. Поэтому рассчитаны лаги прохождения пика весеннего половодья:

| Участок | Практическая интерпретация |
|---|---|
| Ленск → Якутск | около 4 суток до прихода пика |
| Олёкминск → Якутск | около 4 суток до прихода пика |
| Якутск → Жиганск | около 4 суток для нижнего течения |

Упрощенная прогнозная формула для демонстрации:

$$H_{Yakutsk} = -120 + 0.38H_{Lensk} + 0.42H_{Olekminsk}$$

Пороговые уровни: **700 см** предупреждение, **850 см** критический режим.


In [ ]:
def predict_yakutsk_peak(h_lensk, h_olekminsk):
    return -120.0 + 0.38 * h_lensk + 0.42 * h_olekminsk

def hydro_status(level_cm):
    if level_cm >= 850:
        return "CRITICAL"
    if level_cm >= 700:
        return "WARNING"
    return "NORMAL"

examples = [
    ("Пример 1", 750, 760),
    ("Пример 2", 1100, 1050),
    ("Пример 3", 1250, 1250),
]

for title, lensk, olek in examples:
    peak = predict_yakutsk_peak(lensk, olek)
    print(f"{title}: Ленск {lensk} см, Олёкминск {olek} см -> прогноз пика Якутска {peak:.0f} см -> {hydro_status(peak)}")
print("Практический смысл лагов: пик у Ленска/Олёкминска дает около 4 суток на подготовку Якутска.")


Пример 1: Ленск 750 см, Олёкминск 760 см -> прогноз пика Якутска 484 см -> NORMAL
Пример 2: Ленск 1100 см, Олёкминск 1050 см -> прогноз пика Якутска 739 см -> WARNING
Пример 3: Ленск 1250 см, Олёкминск 1250 см -> прогноз пика Якутска 880 см -> CRITICAL
Практический смысл лагов: пик у Ленска/Олёкминска дает около 4 суток на подготовку Якутска.


![Лаги прохождения паводковой волны](plots/Lena_River_Peak_Propagation.png)


## 9. Интегральный балл риска `PSRS`

`PSRS` переводит физические параметры в единую шкалу 0-10:

$$PSRS = clip(5.0 + 1.6 \cdot (0.4Z_{ALT} + 0.3Z_{T320} + 0.2Z_{snow} + 0.1Z_{ROS}), 0, 10)$$

| Балл | Уровень | Управленческий смысл |
|---:|---|---|
| `< 4` | низкий | плановый мониторинг |
| `4-7` | умеренный | повышенная готовность, инженерная профилактика |
| `≥ 7` | высокий | режим ЧС, оперативный штаб, эвакуационные и аварийные мероприятия |


In [ ]:
def stefan_classic(ddt, e=4.3448):
    return 0.0 if ddt < 0 else e * math.sqrt(ddt)

def stefan_hybrid(ddt, h_snow, e_base=4.1866, beta=0.00155):
    return 0.0 if ddt < 0 else e_base * math.sqrt(ddt) * (1.0 + beta * h_snow)

def alt_regression(ddt, ddf, h_snow, p_summer):
    return 197.99 + 0.0096 * ddt - 0.0051 * ddf + 0.370 * h_snow - 0.0045 * p_summer

PSRS_STATS = {
    "alt": {"mean": 221.5, "std": 12.8},
    "t320": {"mean": -0.58, "std": 0.32},
    "snow": {"mean": 35.4, "std": 8.6},
    "ros": {"mean": 1.6, "std": 1.1},
}

def z_score(value, name):
    return (value - PSRS_STATS[name]["mean"]) / PSRS_STATS[name]["std"]

def psrs(alt, t320, h_snow, ros_days):
    weighted_z = (
        0.4 * z_score(alt, "alt") +
        0.3 * z_score(t320, "t320") +
        0.2 * z_score(h_snow, "snow") +
        0.1 * z_score(ros_days, "ros")
    )
    return max(0.0, min(10.0, 5.0 + 1.6 * weighted_z))

def dss_level(score):
    if score < 4:
        return "низкий риск, плановый мониторинг"
    if score < 7:
        return "умеренный риск, режим повышенной готовности"
    return "высокий риск, режим ЧС/оперативный штаб"

scenarios = [
    ("средние исторические условия", 2000, 4000, 35, 150, -0.58, 2),
    ("теплое лето и высокий снег", 2480, 3800, 55, 170, -0.25, 4),
    ("экстремальное протаивание и ROS", 3000, 3400, 70, 180, -0.05, 7),
]

for name, ddt, ddf, snow, precip, t320, ros in scenarios:
    alt_c = stefan_classic(ddt)
    alt_h = stefan_hybrid(ddt, snow)
    alt_r = alt_regression(ddt, ddf, snow, precip)
    score = psrs(alt_h, t320, snow, ros)
    print(f"Сценарий: {name}")
    print(f"  ALT classic: {alt_c:.1f} см; ALT hybrid: {alt_h:.1f} см; ALT regression: {alt_r:.1f} см")
    print(f"  PSRS: {score:.2f} -> {dss_level(score)}")


Сценарий: средние исторические условия
  ALT classic: 194.3 см; ALT hybrid: 197.4 см; ALT regression: 209.1 см
  PSRS: 3.84 -> низкий риск, плановый мониторинг
Сценарий: теплое лето и высокий снег
  ALT classic: 216.4 см; ALT hybrid: 226.3 см; ALT regression: 222.0 см
  PSRS: 6.81 -> умеренный риск, режим повышенной готовности
Сценарий: экстремальное протаивание и ROS
  ALT classic: 238.0 см; ALT hybrid: 254.2 см; ALT regression: 234.5 см
  PSRS: 9.50 -> высокий риск, режим ЧС/оперативный штаб


![Матрица решений МЧС](plots/MCHS_Decision_Matrix.png)


## 10. Что получилось как научный и практический результат

**Научный результат:** построена связанная модель, которая объединяет мерзлотные, климатические и гидрологические факторы. Важный вклад состоит в том, что риск рассматривается не по одному параметру, а как система: сезонное протаивание, глубинное потепление, снеговая теплоизоляция, ROS-события и паводковая волна.

**Практический результат:** разработана логика СППР. Она позволяет перейти от наблюдений к конкретному режиму реагирования: плановый мониторинг, повышенная готовность или режим ЧС.

**Окно упреждения:** гидрологические лаги дают порядка 4 суток для подготовки Якутска при прохождении пика по верхним постам Лены.


## 11. Готовый тезис для защиты

> ВКР решает задачу планирования реакции на риски деградации вечной мерзлоты и связанных гидрологических ЧС. Для этого подготовлены данные наблюдений, построены модели сезонного протаивания и глубинного потепления мерзлоты, рассчитаны лаги прохождения паводковой волны по Лене и создан интегральный балл `PSRS`. Полученная система поддержки принятия решений переводит расчетные показатели в понятные уровни реагирования для органов власти и спасательных служб.


## 12. Ограничения и что важно проговорить комиссии

1. Модели являются инструментом поддержки принятия решений, а не заменой инженерного обследования конкретного здания или участка.
2. Линейный тренд глубинной температуры корректен как первый сценарный ориентир; для проектных решений требуется регулярное обновление по новым скважинным измерениям.
3. Пороговые значения 700 и 850 см используются как демонстрационная шкала реагирования и должны уточняться по официальным муниципальным регламентам и актуальным гидропостам.
4. Сильная сторона работы: все промежуточные таблицы, формулы и графики сохранены в каталоге результатов, поэтому выводы можно проверить и воспроизвести.
